In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession(".cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 51.20,
    "longitude": 4.42,
    "start_date": "2023-10-01",
    "end_date": "2025-12-31",
    "hourly": ["shortwave_radiation", "diffuse_radiation", "direct_normal_irradiance"],
    "timezone": "Europe/Berlin",
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_shortwave_radiation = hourly.Variables(0).ValuesAsNumpy()
hourly_diffuse_radiation = hourly.Variables(1).ValuesAsNumpy()
hourly_direct_normal_irradiance = hourly.Variables(2).ValuesAsNumpy()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    )
}

hourly_data["shortwave_radiation"] = hourly_shortwave_radiation
hourly_data["diffuse_radiation"] = hourly_diffuse_radiation
hourly_data["direct_normal_irradiance"] = hourly_direct_normal_irradiance

hourly_dataframe = pd.DataFrame(data=hourly_data)
print("\nHourly data\n", hourly_dataframe)

In [ ]:
hourly_dataframe

In [ ]:
hourly_dataframe.set_index("date", inplace=True)

In [ ]:
hourly_dataframe = hourly_dataframe.tz_convert("Europe/Brussels")

In [ ]:
hourly_dataframe.index = hourly_dataframe.index.map(lambda x: x - pd.Timedelta(hours=1))

In [ ]:
hourly_dataframe

In [ ]:
hourly_dataframe.to_parquet("data/openmeteo_hourly_antwerpen.parquet.gz", compression="gzip")